# Blog 11 — Databricks Jobs & Pipelines

## Practical Orchestration for Data Engineers

> **Goal:** understand Jobs and Pipelines by actually building and running small workloads, not just reading about them.

This notebook combines:

- concise theory
- notebook-based workload preparation
- Job design
- multi-task dependencies
- parameters
- scheduling concepts
- retries
- monitoring
- Unity Catalog integration
- Medallion workflow design
- Pipeline concepts

The practical exercises are deliberately small so the focus stays on **orchestration**, not complex data engineering logic.

# 1. What are we building?

We will use a simple workflow:

```text
                    JOB
                     │
                     ▼
              Task 1 — Bronze
                     │
                     ▼
              Task 2 — Quality
                     │
                     ▼
              Task 3 — Silver
                     │
                     ▼
               Task 4 — Gold
```

The actual data transformations are intentionally simple.

The objective is to learn:

```text
Notebook
   ↓
Task
   ↓
Dependency
   ↓
Job
   ↓
Run
   ↓
Monitoring
```

# 2. Job vs Notebook

A notebook contains the workload.

A Job orchestrates the workload.

```text
Notebook
    ↓
contains code

Task
    ↓
executes the notebook

Job
    ↓
organizes tasks

Run
    ↓
one execution of the Job
```

A production Data Engineer normally separates:

```text
Business logic
        +
Execution configuration
        +
Orchestration
```

# 3. Practical setup — use a small UC table

For the exercises, we will create a tiny Delta table.

> **Important:** replace the catalog/schema below only if your workspace uses different names.

We use a Unity Catalog table so Blog 11 connects directly to Blog 10.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "medallion_project"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.blog11_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.blog11_silver"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.blog11_gold"

print("Bronze:", BRONZE_TABLE)
print("Silver:", SILVER_TABLE)
print("Gold  :", GOLD_TABLE)


Bronze: workspace.medallion_project.blog11_bronze
Silver: workspace.medallion_project.blog11_silver
Gold  : workspace.medallion_project.blog11_gold


# 4. Create the Bronze workload

This notebook cell represents what a **Bronze task** might do.

In a real project, this could be Auto Loader or another ingestion process.

For Blog 11, we deliberately keep the workload simple because we are learning **orchestration**, not ingestion.

In [0]:
bronze_data = [
    (1001, 501, 1250.50, "COMPLETED"),
    (1002, 502, 850.00, "PENDING"),
    (1003, 503, 420.75, "COMPLETED"),
    (1004, 504, 990.00, "CANCELLED"),
    (1005, 505, 1750.25, "COMPLETED")
]

bronze_df = spark.createDataFrame(
    bronze_data,
    ["order_id", "customer_id", "amount", "status"]
)

(
    bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(BRONZE_TABLE)
)

print("Bronze workload completed.")
display(spark.table(BRONZE_TABLE))


Bronze workload completed.


order_id,customer_id,amount,status
1001,501,1250.5,COMPLETED
1002,502,850.0,PENDING
1003,503,420.75,COMPLETED
1004,504,990.0,CANCELLED
1005,505,1750.25,COMPLETED


# 5. Create the Silver workload

The Silver task depends on Bronze.

A simple transformation:

```text
Bronze
  ↓
keep valid orders
  ↓
Silver
```

This is deliberately simple. The important part later will be the **dependency between the tasks**.

In [0]:
silver_df = (
    spark.table(BRONZE_TABLE)
        .filter(F.col("amount") > 0)
        .withColumn(
            "amount_band",
            F.when(F.col("amount") >= 1000, "HIGH")
             .otherwise("STANDARD")
        )
)

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_TABLE)
)

print("Silver workload completed.")
display(spark.table(SILVER_TABLE))


Silver workload completed.


order_id,customer_id,amount,status,amount_band
1001,501,1250.5,COMPLETED,HIGH
1002,502,850.0,PENDING,STANDARD
1003,503,420.75,COMPLETED,STANDARD
1004,504,990.0,CANCELLED,STANDARD
1005,505,1750.25,COMPLETED,HIGH


# 6. Create the Gold workload

The Gold task depends on Silver.

Here we create a small business aggregation:

```text
Silver
   ↓
group by status
   ↓
Gold
```

In [0]:
gold_df = (
    spark.table(SILVER_TABLE)
        .groupBy("status")
        .agg(
            F.count("*").alias("order_count"),
            F.sum("amount").alias("total_amount")
        )
)

(
    gold_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_TABLE)
)

print("Gold workload completed.")
display(spark.table(GOLD_TABLE))


Gold workload completed.


status,order_count,total_amount
COMPLETED,3,3421.5
PENDING,1,850.0
CANCELLED,1,990.0


# 7. What we have created

We now have four logical workloads:

```text
Bronze workload
      ↓
Silver workload
      ↓
Gold workload
```

and a validation workload can sit alongside them.

The important point:

> These workloads can now be turned into separate **Job Tasks**.

A Job does not replace the notebooks.

It **orchestrates their execution**.

# 8. Practical Lab — create the Job

Now move from notebook execution to Databricks Job configuration.

In the Databricks workspace:

```text
Workflows
   ↓
Jobs
   ↓
Create Job
```

Create a Job named:

```text
Blog 11 — Orders Pipeline
```

Add the first task:

```text
Task key: bronze_task
Type: Notebook
```

Point it to the notebook containing the Bronze workload.

### Expected structure

```text
Orders Pipeline
      │
      └── bronze_task
```

# 9. Practical Lab — add the Silver task

Add another task:

```text
Task key: silver_task
Type: Notebook
```

Then configure:

```text
Depends on:
bronze_task
```

Your Job graph should now look like:

```text
bronze_task
     │
     ▼
silver_task
```

This is the first important hands-on Job concept:

> **Task dependencies determine execution order.**

# 10. Practical Lab — add the Gold task

Add:

```text
Task key: gold_task
Type: Notebook
```

Set:

```text
Depends on:
silver_task
```

Now the graph becomes:

```text
bronze_task
     │
     ▼
silver_task
     │
     ▼
gold_task
```

You have created a real multi-task workflow.

# 11. Practical Lab — add a validation task

Create another notebook containing a simple validation.

For example:

```python
assert spark.table(BRONZE_TABLE).count() == 5
assert spark.table(SILVER_TABLE).count() == 5
assert spark.table(GOLD_TABLE).count() > 0

print("Validation passed.")
```

Create a Job task:

```text
Task key:
validation_task
```

Set:

```text
Depends on:
gold_task
```

Now:

```text
bronze
  ↓
silver
  ↓
gold
  ↓
validation
```

This is much closer to a real production workflow.

# 12. Run the Job

Now click:

```text
Run now
```

The Job should execute tasks according to the dependency graph.

Expected sequence:

```text
bronze_task
     ↓
silver_task
     ↓
gold_task
     ↓
validation_task
```

The key thing to observe is **not the data transformation**.

Observe:

- task order
- task status
- dependency behavior
- execution duration
- overall Job status

# 13. Understanding the Job graph

A successful run should conceptually look like:

```text
bronze_task       SUCCESS
      │
      ▼
silver_task       SUCCESS
      │
      ▼
gold_task         SUCCESS
      │
      ▼
validation_task   SUCCESS
```

This graph is the core reason Jobs are useful.

The platform understands the workflow rather than treating each notebook as an isolated execution.

# 14. Practical Lab — parameters

Now make the workload reusable.

Create a parameter:

```text
process_date
```

Example value:

```text
2026-08-24
```

The Job passes:

```text
process_date = 2026-08-24
```

to the notebook.

The notebook can then use that value to control processing.

The key concept is:

```text
Same notebook
      +
Different parameter
      ↓
Different execution
```

In [0]:
# Example parameter-aware notebook logic

# In an actual Databricks Job, the parameter can be
# provided through the Job configuration.

# Example conceptual value:
process_date = "2026-08-24"

print("Processing date:", process_date)

# A real workload could then use it:
#
# df = spark.sql(f'''
#     SELECT *
#     FROM workspace.medallion_project.orders
#     WHERE order_date = '{process_date}'
# ''')


Processing date: 2026-08-24


# 15. Why parameters are useful

Without parameters:

```text
orders_2026_08_24 notebook
orders_2026_08_25 notebook
orders_2026_08_26 notebook
```

With parameters:

```text
One notebook
     +
process_date
     ↓
Reusable workload
```

This is especially useful for:

- daily processing
- backfills
- testing
- environment-specific execution
- rerunning historical dates

# 16. Practical Lab — scheduling

A Job can be scheduled.

In the Job configuration, configure a schedule such as:

```text
Daily
```

or another appropriate frequency.

Conceptually:

```text
Schedule
   ↓
Job starts
   ↓
Bronze
   ↓
Silver
   ↓
Gold
   ↓
Validation
```

For this learning exercise, you do not need to leave the schedule running indefinitely.

The objective is understanding where scheduling belongs in the architecture.

# 17. Practical Lab — retries

Open a task's configuration and examine the retry settings.

Conceptually:

```text
Maximum retries = 2
```

If the task encounters a retryable failure:

```text
Attempt 1 → Failed
Attempt 2 → Retry
Attempt 3 → Success
```

### Important

Retries are for transient problems.

A permanent SQL or logic error should be fixed rather than hidden behind repeated retries.

# 18. Practical Lab — timeout

Inspect the task timeout configuration.

Conceptually:

```text
Task
 ↓
maximum execution duration
 ↓
timeout
```

Timeouts are useful when a workload unexpectedly runs much longer than intended.

This is an operational safeguard, not a replacement for performance optimization.

# 19. Practical Lab — monitoring a run

Open the Job run you just created.

Inspect:

```text
Run status
Task status
Start time
Duration
Dependencies
Logs
```

For example:

```text
Run
 │
 ├── Bronze       SUCCESS
 ├── Silver       SUCCESS
 ├── Gold         SUCCESS
 └── Validation   SUCCESS
```

This is the practical side of Job monitoring.

# 20. Practical Lab — understand a failure path

You do **not** need to build a complicated failure-injection experiment.

Instead, understand the graph:

```text
Bronze
  ↓
Silver FAILED
  ↓
Gold does not receive a successful dependency
  ↓
Validation does not proceed normally
```

The important lesson:

> Dependencies are also part of failure handling.

The detailed failure-recovery mechanics were already covered in Blog 9.

# 21. Jobs + Unity Catalog

Your Job is now executing against governed UC objects:

```text
workspace.medallion_project.blog11_bronze
workspace.medallion_project.blog11_silver
workspace.medallion_project.blog11_gold
```

This gives you the distinction:

```text
Unity Catalog
      ↓
governs data assets

Job
      ↓
orchestrates workloads
```

The two concepts complement each other.

# 22. Jobs + Medallion Architecture

Our practical Job is essentially a small Medallion workflow:

```text
             JOB
              │
              ▼
           BRONZE
              │
              ▼
           SILVER
              │
              ▼
            GOLD
              │
              ▼
         VALIDATION
```

This is the pattern you should eventually reproduce in the final end-to-end project.

# 23. What is a Databricks Pipeline?

A Pipeline is a managed data-processing workflow.

Modern Databricks terminology includes **Lakeflow Declarative Pipelines** for declarative data processing.

For this blog, keep the distinction simple:

```text
Job
 ↓
orchestrates workloads

Pipeline
 ↓
manages data-processing flow
```

We are not going deep into advanced Lakeflow features here.

# 24. Job vs Pipeline

| | Job | Pipeline |
|---|---|---|
| Main purpose | Orchestration | Data processing |
| Tasks | Central concept | Data flow is central |
| Scheduling | Yes | Can be orchestrated |
| Dependencies | Explicit task graph | Data dependencies |
| Notebook execution | Common | Not the main abstraction |
| ETL | Can orchestrate ETL | Core purpose |

A Job and a Pipeline can work together.

They are not simply competing alternatives.

# 25. A practical production pattern

A realistic architecture could be:

```text
                 JOB
                  │
       ┌──────────┼──────────┐
       ▼          ▼          ▼
    Ingestion   Quality   Reference
       │
       ▼
    Pipeline
       │
       ▼
     Bronze
       │
       ▼
     Silver
       │
       ▼
      Gold
       │
       ▼
   Validation
```

And:

```text
Unity Catalog → governance
Jobs          → orchestration
Pipeline      → managed data processing
Delta         → transactional storage
Data Quality  → correctness
```

# 26. What you should actually practice

Before considering Blog 11 complete, you should be able to perform these actions in Databricks:

### Hands-on checklist

- [ ] Create a small notebook workload
- [ ] Create a Job
- [ ] Add a notebook task
- [ ] Add a second task
- [ ] Configure a dependency
- [ ] Add a third task
- [ ] Run the multi-task Job
- [ ] Inspect the Job graph
- [ ] Inspect a Job run
- [ ] Configure a parameter
- [ ] Understand scheduling configuration
- [ ] Inspect retry settings
- [ ] Inspect timeout settings
- [ ] Understand failure propagation
- [ ] Connect Job tasks to Unity Catalog tables

This is the practical competency Blog 11 is designed to build.

# 27. What you should know after Blog 11

You should be able to explain and demonstrate:

### Job

> An orchestration object that schedules and runs workloads.

### Task

> An executable unit inside a Job.

### Dependency

> Controls the relationship and execution order between tasks.

### Parameter

> Runtime configuration passed into a workload.

### Retry

> Allows configured tasks to retry after failures.

### Timeout

> Limits how long a task is allowed to run.

### Run

> One execution of a Job.

### Monitoring

> Observing task and Job execution status, duration and logs.

### Pipeline

> A managed data-processing workflow.

Most importantly, you should have **actually created and run a multi-task Job**, not just read about one.

# 28. Final mental model

```text
                    DATABRICKS JOB
                           │
                 ┌─────────┴─────────┐
                 ▼                   ▼
             Task 1               Task 2
                 │                   │
                 └─────────┬─────────┘
                           ▼
                         Task 3
                           │
                           ▼
                       Validation
```

The surrounding platform:

```text
Unity Catalog
      ↓
governs data

Job
      ↓
orchestrates tasks

Pipeline
      ↓
manages data processing

Delta
      ↓
stores transactional data
```

### One sentence to remember

> **A Databricks Job turns individual workloads into an automated, scheduled, dependency-aware and monitored workflow.**

This gives you the practical foundation needed before moving to **Blog 12 — Auto Loader + Structured Streaming**.